# Model Training & Evaluation

## Objective

The objective of this notebook is to:

- Load the training and testing datasets.
- Apply the preprocessing pipeline developed during feature engineering.
- Train multiple machine learning models.
- Evaluate model performance using classification metrics.
- Compare baseline models and identify top-performing candidates for hyperparameter tuning.

In [15]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.pipeline import Pipeline

## Load Training and Testing Data

In [2]:
train_df = pd.read_csv('../artifacts/train.csv')
test_df = pd.read_csv('../artifacts/test.csv')

print(train_df.shape)
print(test_df.shape)

(1719, 33)
(430, 33)


## Load Preprocessing Pipeline

In [3]:
preprocessor = joblib.load(
    '../artifacts/preprocessor.pkl'
)

In [5]:
X_train = train_df.drop('Diagnosis', axis=1)
y_train = train_df['Diagnosis']

X_test = test_df.drop('Diagnosis', axis=1)
y_test = test_df['Diagnosis']

## Apply Data Transformations

In [6]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(1719, 38)
(430, 38)


In [29]:
models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "KNN":
        KNeighborsClassifier(),

    "SVC":
        SVC(probability=True),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(random_state=42),

    "AdaBoost":
        AdaBoostClassifier(random_state=42),

    "Gradient Boosting":
        GradientBoostingClassifier(random_state=42),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric='logloss'
        ),

    "CatBoost":
        CatBoostClassifier(
            random_state=42,
            verbose=0
        ),

    "LightGBM":
        LGBMClassifier(
            random_state=42,
            verbose=-1
        )
}

In [30]:
results = []

for name, model in models.items():

    model.fit(
        X_train_processed,
        y_train
    )

    y_pred = model.predict(
        X_test_processed
    )

    y_prob = model.predict_proba(
        X_test_processed
    )[:,1]

    results.append({

        "Model": name,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Precision":
            precision_score(
                y_test,
                y_pred
            ),

        "Recall":
            recall_score(
                y_test,
                y_pred
            ),

        "F1 Score":
            f1_score(
                y_test,
                y_pred
            ),

        "ROC AUC":
            roc_auc_score(
                y_test,
                y_prob
            )
    })

In [31]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='ROC AUC',
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
9,LightGBM,0.953488,0.945946,0.921053,0.933333,0.951936
6,Gradient Boosting,0.944186,0.921053,0.921053,0.921053,0.947321
7,XGBoost,0.941860,0.931973,0.901316,0.916388,0.945239
8,CatBoost,0.948837,0.933333,0.921053,0.927152,0.943156
5,AdaBoost,0.918605,0.872611,0.901316,0.886731,0.938399
4,Random Forest,0.939535,0.943662,0.881579,0.911565,0.936423
2,SVC,0.853488,0.844961,0.717105,0.775801,0.904096
0,Logistic Regression,0.816279,0.744966,0.730263,0.737542,0.886478
3,Decision Tree,0.876744,0.807453,0.855263,0.830671,0.871876
1,KNN,0.760465,0.720721,0.526316,0.608365,0.761253


In [ ]:
# Prints key classification metrics in a clean format.



def classification_report_print(y_true, y_pred, model_name="Model"):
    """
    Prints key classification metrics in a clean format.

    Parameters:
    - y_true: Actual labels
    - y_pred: Predicted labels
    - model_name: Name of the model (optional)
    """

    print(f"=== {model_name} Performance Metrics ===")
    print(f"Accuracy      : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision     : {precision_score(y_true, y_pred):.4f}")
    print(f"Recall        : {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score      : {f1_score(y_true, y_pred):.4f}")

# Plots a clean confusion matrix heatmap.
def confusion_matrix_print(y_true, y_pred, model_name="Model"):
    """
    Plots a clean confusion matrix heatmap.

    Parameters:
    - y_true: Actual labels
    - y_pred: Predicted labels
    - model_name: Name of the model (optional)
    """

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", linewidths=0.5)
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

# Plots the ROC curve and prints the AUC score.
def plot_roc_curve(y_true, y_proba, model_name="Model"):
    """
    Plots the ROC curve and prints the AUC score.

    Parameters:
    - y_true: Actual labels (0/1)
    - y_proba: Predicted probabilities for the positive class
    - model_name: Name of the model (string)
    """

    auc = roc_auc_score(y_true, y_proba)
    fpr, tpr, _ = roc_curve(y_true, y_proba)

    plt.figure(figsize=(6,4))
    plt.plot(fpr, tpr, label=f"{model_name} AUC = {auc:.3f}", linewidth=2)
    plt.plot([0,1], [0,1], 'k--', linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Plots the Precision–Recall curve and prints the Average Precision (AP) score.
def plot_precision_recall_curve(y_true, y_proba, model_name="Model"):
    """
    Plots the Precision–Recall curve and prints the Average Precision (AP) score.

    Parameters:
    - y_true: Actual labels (0/1)
    - y_proba: Predicted probabilities for the positive class
    - model_name: Name of the model (string)
    """

    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)

    plt.figure(figsize=(6,4))
    plt.plot(recall, precision, label=f"{model_name} AP = {ap:.3f}", linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall Curve — {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def run_pipeline(model, preprocessor, X_train, X_test, y_train):
    """
    Fits a pipeline for ANY model and returns predictions + probabilities.
    """

    pipe = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    return pipe, y_pred, y_proba



